# 3 · PE + SFT + RL (GRPO)
Starts from the SFT model of notebook 2 (needs `sft/adapter` and `sft/samples.jsonl` on Drive) and adds a second LoRA trained with GRPO. Same frozen eval protocol.

**What makes this RL stage different from last time**
| Last time | Now |
|---|---|
| reward compared the answer to the *whole solution text* (almost never matched) | reward = **the evaluation grader itself** on the final `\boxed{}` |
| Qwen3-4B-**Base** + custom `<SOLUTION>` tags | the **same Thinking model** as the baseline, same prompts, same `\boxed{}` format |
| random problems (many groups all-right or all-wrong → zero signal) | problems the model solves **1–3 of 4 times**, **re-checked with the SFT model** (the model RL starts from), + long-but-correct ones, + option-shuffled MCQs; MMLU-Pro capped at 35% so RL stays on your own question types |
| truncation ignored | rollouts get **16,384 tokens, the same budget as eval**; an answer cut off there also fails at eval, so it gets −0.5 |
| Unsloth default `bnpo` loss | `dr_grpo` (no length bias), no KL (β=0), clip-higher ε=0.28, 8 samples × 4 problems per step (a bigger, less noisy batch costs little extra because generation time is set by the longest answer), LoRA learning rate 1e-5 |

Reward: **+1** correct · **0–0.5** partial credit on multi-blank · **0** wrong · **−0.5** no final `\boxed{}` answer or cut off at 16k.
We keep the step-20 and step-40 adapters and the final one, pick the best on **dev**, and only then score **test**. Every 10 steps the log prints a `HEALTH` line (accuracy, cut-offs, groups without signal, min/step) with a warning if something looks wrong, so you can stop early instead of paying for a broken run.

**Run on an A100 80GB** (16 rollouts × 16k tokens don't fit in 40GB; the notebook checks). Nothing to edit: Runtime → Run all.
**Cost (≈ 7.5 units/h):** merge ~5 min · re-check ≤ 12 units · GRPO ≤ 55 units (estimated 6–9 min/step with 32 answers per step → roughly 45–70 steps; it prints real `min/step` and stops cleanly at the limit; constant learning rate, so stopping early is clean) · dev/test evals ~8 units · 3 extra test samples for all three models ~14 units → **≈ 87–92 units**.
The extra test samples make every row of the final table an average of 4 answers per question, which roughly halves the noise in the comparison.

In [ ]:
#@title Settings (edit here only)
USE_DRIVE = True          # keep True: every stage saves to Drive and resumes after a disconnect
WORK = "/content/drive/MyDrive/qwen_math_ft" if USE_DRIVE else "/content/qwen_math_ft"
CODE = "/content/qwen_ft"
# The plan is fixed; nothing here needs editing.
GRPO_STEPS = 120
GRPO_LR = 1e-5
PROMPTS_PER_STEP = 4       # 4 problems x 8 answers per step
MAX_COMPLETION = 16384     # RL rollout budget = eval budget
CUT_OFF_ROLLOUTS = "penalize"   # a rollout cut off at 16k also fails at eval -> -0.5
GRPO_GPU_UTIL = 0.85       # retried automatically at 0.70 if GRPO runs out of GPU memory
RECHECK_UNITS = 12         # compute-unit limit for re-checking RL prompts with the SFT model
GRPO_UNITS = 55            # compute-unit limit for GRPO training
EXTRA_TEST_SAMPLES = 3     # extra answers per test question for all three models (final table = average of 4)
MERGED = "/content/sft_merged"   # local disk; rebuilt in ~5 min if the runtime resets
import os, sys, subprocess, json, shutil
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
for d in [WORK, f"{WORK}/data", f"{WORK}/results", f"{WORK}/sft", f"{WORK}/grpo", f"{CODE}/mathrl", f"{CODE}/tests"]:
    os.makedirs(d, exist_ok=True)
DATA = f"{WORK}/data/public.jsonl"
BASELINE = f"{WORK}/data/starter_results_backup.jsonl"   # optional: old baseline responses, rescored for the table
# Data files are read from Drive (Colab's upload button fails on files of a few MB).
# Looks in MyDrive/qwen_math_ft/data/, then MyDrive/qwen_math_ft/, MyDrive/, and /content/ (the Files panel).
for name, dst in [("public.jsonl", DATA), ("starter_results_backup.jsonl", BASELINE)]:
    if not os.path.exists(dst):
        for src in [f"{WORK}/{name}", f"{os.path.dirname(WORK)}/{name}", f"/content/{name}"]:
            if os.path.exists(src):
                shutil.copy(src, dst); print(f"copied {src} -> {dst}"); break
if not os.path.exists(DATA):
    raise FileNotFoundError(
        "public.jsonl not found. Put it (and optionally starter_results_backup.jsonl) in Google Drive at\n"
        f"    {WORK}/data/   (i.e. My Drive > qwen_math_ft > data)\n"
        "or drag it into the Files panel on the left (it lands in /content), then re-run this cell.")
print("data:", DATA, "| baseline file:", "found" if os.path.exists(BASELINE) else "not found (baseline row will be skipped)")
os.environ.update(PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")

class StepFailed(RuntimeError):
    def __init__(self, tail):
        last = "".join(tail.splitlines(True)[-25:])
        super().__init__("Step failed. Last lines of its log:\n\n" + last + "\nRe-running this cell resumes where it stopped.")
        self.oom = "out of memory" in tail.lower()

def run(cmd, done_if=None):
    """Run one pipeline step as its own process (frees all GPU memory when it ends), streaming its log."""
    if done_if and os.path.exists(done_if):
        print(f"[skip] already done -> {done_if}"); return
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail.append(line)
        if len(tail) > 200: del tail[:100]
    if p.wait() != 0:
        raise StepFailed("".join(tail))

def summary(path):
    s = path.replace(".jsonl", "_summary.json")
    return json.load(open(s)) if os.path.exists(s) else None

def gpu_units_per_hour():
    """Colab compute units per hour for this GPU. Google doesn't publish these; third-party measurements, Mar 2026."""
    try:
        q = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits", shell=True,
                           capture_output=True, text=True).stdout.strip().splitlines()[0]
        name, mem = q.rsplit(",", 1); mem = float(mem)
    except Exception:
        return None, "no GPU"
    if "A100" in name: return (7.52 if mem > 60000 else 5.40), name.strip()
    if "L4" in name: return 1.71, name.strip()
    if "T4" in name: return 1.19, name.strip()
    return None, name.strip()
RATE, GPU_NAME = gpu_units_per_hour()

def hours_for(units):
    """Turn a compute-unit limit into GPU hours on this GPU (unknown GPU -> assume the pricier A100 80GB rate)."""
    return round(units / (RATE or 7.52), 2)
print("WORK =", WORK, "| GPU:", GPU_NAME, f"(~{RATE} units/h)" if RATE else "(rate unknown)")

In [ ]:
#@title Install pinned packages (~5-8 min, once per runtime)
# These exact versions were checked against each other's requirements and support Colab's Python 3.13:
#   vllm 0.15.1 needs torch 2.9.1 + transformers <5; unsloth 2026.5.8 needs peft>=0.18, trl<=0.24, torch<2.11.
#   peft is pinned to 0.18.1 on purpose: peft>=0.19 imports EmbeddingParallel from transformers, which
#   breaks checkpoint resume with this transformers version (the error from last time).
#   torchao is pinned to 0.15.0, the release built for torch 2.9.1 (0.17+ needs torch 2.11 and breaks vLLM's import).
#   Every package is pinned, so a new release can't change the environment between runs.
# Any failure here stops "Run all" and prints the real error.
import subprocess, sys
def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        print(out[-6000:])
        raise RuntimeError(f"Install step failed (exit code {r.returncode}). The error is printed just above.")
    return out
print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))
sh('pip install -q "torch==2.9.1" "vllm==0.15.1" "xformers==0.0.33.post2" "transformers==4.57.6" "trl==0.22.2" '
   '"peft==0.18.1" "unsloth==2026.5.8" "unsloth_zoo==2026.5.4" "datasets==4.3.0" "accelerate==1.15.0" '
   '"bitsandbytes==0.50.2" "sympy==1.14.0" "torchao==0.15.0"')
# Verify in a separate process (this notebook's own Python never imports torch, so no restart is needed).
VERIFY = (
    "import unsloth\n"
    "import torch, vllm, transformers, trl, peft, datasets, torchao\n"
    "from vllm import LLM, SamplingParams\n"
    "import transformers.quantizers.quantizer_torchao\n"
    "v = {'torch': torch.__version__.split('+')[0], 'vllm': vllm.__version__, 'transformers': transformers.__version__,\n"
    "     'trl': trl.__version__, 'peft': peft.__version__, 'datasets': datasets.__version__, 'torchao': torchao.__version__}\n"
    "exp = {'torch': '2.9.1', 'vllm': '0.15.1', 'transformers': '4.57.6', 'trl': '0.22.2', 'peft': '0.18.1', 'datasets': '4.3.0', 'torchao': '0.15.0'}\n"
    "bad = {k: (v[k], exp[k]) for k in exp if v[k] != exp[k]}\n"
    "assert not bad, f'version mismatch {bad}'\n"
    "assert torch.cuda.is_available(), 'No GPU - Runtime > Change runtime type > A100'\n"
    "assert torch.cuda.is_bf16_supported(), 'This GPU has no bfloat16 (T4/V100). Use an A100.'\n"
    "print(v, '| GPU:', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9), 'GB')\n")
open("/tmp/verify_env.py", "w").write(VERIFY)
print(sh(f"{sys.executable} /tmp/verify_env.py"))
print("install OK")

## Project code
The same files are written by all three notebooks, so prompts, data split and grader are identical. Run these cells; no need to edit.

In [ ]:
%%writefile /content/qwen_ft/mathrl/__init__.py

In [ ]:
%%writefile /content/qwen_ft/mathrl/common.py
"""Shared code for every stage: config, prompts, data split, answer extraction, grading.

Every notebook writes this exact file, so prompts / split / grader are identical
across PE, PE+SFT and PE+SFT+RL.
"""
import json, os, random, re, signal, math
from contextlib import contextmanager

# ----------------------------------------------------------------------------
# Frozen evaluation protocol (identical for every model we compare)
# ----------------------------------------------------------------------------
BASE_MODEL = "unsloth/Qwen3-4B-Thinking-2507"   # bf16 copy of Qwen/Qwen3-4B-Thinking-2507
EVAL_MAX_TOKENS = 16384                         # Qwen card recommends >=32k; 16k fits our budget
EVAL_SAMPLING = dict(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0)  # Qwen-recommended
SPLIT_SEED = 151
N_TEST = 250
N_DEV = 125
LORA_RANK = 32

# ----------------------------------------------------------------------------
# Prompt engineering (reused verbatim from the 47.25% run)
# ----------------------------------------------------------------------------
SYSTEM_PROMPT_MCQ = (
    "You are a world-class mathematician. "
    "This is a multiple choice question - select the best answer. "
    "Reason briefly. "
    "You MUST ALWAYS end your response with \\boxed{X} where X is your chosen letter. "
    "Always attempt the problem even if you are unsure. Never say you cannot solve it. "
    "After reaching a conclusion, immediately write \\boxed{} without additional commentary."
)
SYSTEM_PROMPT_MATH = (
    "You are a world-class mathematician. "
    "Solve the problem step by step. "
    "Count the number of [ANS] placeholders in the question - you must provide exactly that many answers. "
    "Separate multiple answers by commas inside a single \\boxed{}, e.g. \\boxed{3, 7}. "
    "You MUST ALWAYS end your response with \\boxed{}. "
    "Always attempt the problem even if you are unsure. Never say you cannot solve it. "
    "After reaching a conclusion, immediately write \\boxed{} without additional commentary."
)
SYSTEM_PROMPT_OTHER = (
    "You are a world-class mathematician. "
    "This question has answer choices embedded in the text. "
    "Reason briefly. "
    "Select the correct letter(s) and put them inside \\boxed{}. "
    "If multiple answers are correct write them together like \\boxed{BC}. "
    "You MUST ALWAYS end your response with \\boxed{}. "
    "Always attempt the problem even if you are unsure. Never say you cannot solve it. "
    "After reaching a conclusion, immediately write \\boxed{} without additional commentary."
)
LETTERS = "ABCDEFGHIJKLMNOP"


# The original baseline's ("starter") prompts, copied verbatim from the baseline notebook (Main1-2.ipynb),
# including its missing spaces between sentences. Only used to re-run the baseline at 16k for a fair comparison.
STARTER_PROMPT_MATH = (
    "You are a world-class mathematician."
    "Put your final answer inside \\boxed{}. "
    "Solve the problem. Then independently verify your answer by checking it from a different angle or plugging it back in."
    "Be very concise"
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)
STARTER_PROMPT_MCQ = (
    "You are a world-class mathematician."
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
    "Be very concise"
)
STARTER_PROMPT_OTHER = (
    "You are a world-class mathematician."
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
    "Be very concise"
    "you can pick multiple answers if they exist"
)
PROMPT_SETS = {
    "pe": (SYSTEM_PROMPT_MCQ, SYSTEM_PROMPT_OTHER, SYSTEM_PROMPT_MATH),
    "starter": (STARTER_PROMPT_MCQ, STARTER_PROMPT_OTHER, STARTER_PROMPT_MATH),
}


def build_messages(question, options=None, prompt_set="pe"):
    """Same routing as the PE notebook (and the baseline notebook): MCQ / embedded choices / math."""
    mcq, other, math_ = PROMPT_SETS[prompt_set]
    if options:
        opts = "\n".join(f"{LETTERS[i]}. {str(o).strip()}" for i, o in enumerate(options))
        return [{"role": "system", "content": mcq},
                {"role": "user", "content": f"{question}\n\nOptions:\n{opts}"}]
    if "A." in question or "B." in question:
        return [{"role": "system", "content": other},
                {"role": "user", "content": question}]
    return [{"role": "system", "content": math_},
            {"role": "user", "content": question}]


def prompt_text(tokenizer, item, prompt_set="pe"):
    """The exact text fed to the model (chat template + generation prompt, ends with '<think>\\n')."""
    return tokenizer.apply_chat_template(build_messages(item["question"], item.get("options"), prompt_set),
                                         tokenize=False, add_generation_prompt=True)


# ----------------------------------------------------------------------------
# Data + split
# ----------------------------------------------------------------------------
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


def gold_list(item):
    a = item["answer"]
    return [str(x) for x in a] if isinstance(a, list) else [str(a)]


def qtype(item):
    if item.get("options"):
        return "mcq"
    return "free_multi" if len(gold_list(item)) > 1 else "free_single"


def make_split(items, seed=SPLIT_SEED, n_test=N_TEST, n_dev=N_DEV):
    """Stratified by question type. Deterministic. Returns dict id -> 'test'|'dev'|'train'."""
    by_type = {}
    for it in sorted(items, key=lambda x: str(x["id"])):
        by_type.setdefault(qtype(it), []).append(it)
    rng = random.Random(seed)
    total = len(items)
    split = {}
    for t, group in sorted(by_type.items()):
        rng.shuffle(group)
        k_test = round(n_test * len(group) / total)
        k_dev = round(n_dev * len(group) / total)
        for i, it in enumerate(group):
            split[it["id"]] = "test" if i < k_test else ("dev" if i < k_test + k_dev else "train")
    return split


def norm_question(q):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", str(q).lower())).strip()


class NearDupIndex:
    """Flags outside questions that are near-copies of ours (reworded, re-spaced, options reordered), not only exact
    matches, so MMLU-Pro / OpenR1 training data can't smuggle in a test question.
      * question text: shares >= half of its 6-word shingles with one of our questions
      * MCQ options: >= 5 identical non-trivial options (>= 4 chars) with one of our questions"""

    def __init__(self, items, n=6):
        self.n, self.idx, self.size, self.opt_idx = n, {}, {}, {}
        for it in items:
            sh = self._shingles(it["question"])
            self.size[it["id"]] = len(sh)
            for x in sh:
                self.idx.setdefault(x, set()).add(it["id"])
            for o in it.get("options") or []:
                k = norm_question(o)
                if len(k) >= 4:
                    self.opt_idx.setdefault(k, set()).add(it["id"])

    def _shingles(self, text):
        w = norm_question(text).split()
        return {" ".join(w[i:i + self.n]) for i in range(max(1, len(w) - self.n + 1))} if w else set()

    def is_dup(self, question, options=None):
        sh = self._shingles(question)
        hits = {}
        for x in sh:
            for i in self.idx.get(x, ()):
                hits[i] = hits.get(i, 0) + 1
        if any(h >= 0.5 * min(len(sh), self.size[i]) for i, h in hits.items()):
            return True
        oh = {}
        for o in options or []:
            k = norm_question(o)
            if len(k) >= 4:
                for i in self.opt_idx.get(k, ()):
                    oh[i] = oh.get(i, 0) + 1
        return any(v >= 5 for v in oh.values())


# ----------------------------------------------------------------------------
# Resumable JSONL output
# ----------------------------------------------------------------------------
def done_keys(path, key="id"):
    if not os.path.exists(path):
        return set()
    out = set()
    with open(path) as f:
        for l in f:
            try:
                out.add(json.loads(l)[key])
            except Exception:
                pass  # a half-written last line after a crash is skipped and redone
    return out


def append_jsonl(path, rows):
    with open(path, "a") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
        f.flush()
        os.fsync(f.fileno())


# ----------------------------------------------------------------------------
# Answer extraction
# ----------------------------------------------------------------------------
def all_boxed(text):
    """Contents of every \\boxed{...} / \\fbox{...} (brace-matched), in order."""
    out = []
    for m in re.finditer(r"\\(?:boxed|fbox)\s*\{", text or ""):
        i = m.end(); depth = 1; j = i
        while j < len(text) and depth:
            if text[j] == "{": depth += 1
            elif text[j] == "}": depth -= 1
            j += 1
        if depth == 0:
            out.append(text[i:j - 1].strip())
    return out


def last_boxed(text):
    b = all_boxed(text)
    return b[-1] if b else None


def final_section(text):
    """Text after the reasoning block (if the model closed it)."""
    return text.split("</think>", 1)[1] if "</think>" in (text or "") else None


def split_top_level(s):
    """Split on top-level ';' if present, else top-level ','. Respects (), [], {}."""
    def _split(sep):
        parts, depth, cur, k = [], 0, [], 0
        while k < len(s):
            ch = s[k]
            if ch in "([{":
                depth += 1
            elif ch in ")]}":
                depth = max(0, depth - 1)
            if ch == sep and depth == 0:
                parts.append("".join(cur)); cur = []
            else:
                cur.append(ch)
            k += 1
        parts.append("".join(cur))
        return [p.strip() for p in parts if p.strip() != ""]
    semi = _split(";")
    return semi if len(semi) > 1 else _split(",")


# ----------------------------------------------------------------------------
# Grading
# ----------------------------------------------------------------------------
REL_TOL = 1e-3      # WeBWorK-style default relative tolerance (0.1%)
ABS_TOL = 1e-8


class _Timeout(Exception):
    pass


@contextmanager
def time_limit(seconds):
    """SIGALRM timeout; silently disabled outside the main thread."""
    try:
        old = signal.signal(signal.SIGALRM, lambda *a: (_ for _ in ()).throw(_Timeout()))
        signal.setitimer(signal.ITIMER_REAL, seconds)
        active = True
    except Exception:
        active = False
    try:
        yield
    finally:
        if active:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old)


def _strip_latex(s):
    s = str(s).strip()
    s = re.sub(r"\\(text|mathrm|textbf|mathbf|operatorname)\s*\{([^{}]*)\}", r"\2", s)
    for a, b in [("\\left", ""), ("\\right", ""), ("\\!", ""), ("\\,", ""), ("\\;", ""), ("\\:", ""),
                 ("\\ ", " "), ("$", ""), ("\\displaystyle", ""), ("\\dfrac", "\\frac"), ("\\tfrac", "\\frac"),
                 ("\\cdot", "*"), ("\\times", "*"), ("\\div", "/"), ("\\pi", " pi "), ("\\infty", " oo "),
                 ("\\leq", "<="), ("\\geq", ">="), ("\\le", "<="), ("\\ge", ">="),
                 ("^\\circ", ""), ("^{\\circ}", ""), ("\\circ", ""),
                 ("\\%", ""), ("%", ""), ("\\{", "("), ("\\}", ")")]:
        s = s.replace(a, b)
    s = s.strip().rstrip(".").strip()
    return s


def _latex_to_expr(s):
    # Every conversion inserts spaces/'*' so 'x\sqrt{2}' never becomes the symbol run 'xsqrt'.
    for _ in range(3):  # nested fractions / roots
        s = re.sub(r"\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}", r" ((\1)/(\2)) ", s)
        s = re.sub(r"\\sqrt\s*\[([^\]]*)\]\s*\{([^{}]*)\}", r" ((\2)**(1/(\1))) ", s)
        s = re.sub(r"\\sqrt\s*\{([^{}]*)\}", r" sqrt(\1) ", s)
    s = re.sub(r"\\sqrt\s*(\d+(\.\d+)?)", r" sqrt(\1) ", s)
    fn = r"(sin|cos|tan|ln|log|exp|arcsin|arccos|arctan|sec|csc|cot|sinh|cosh|tanh)"
    s = re.sub(r"\\" + fn + r"\s*\^\s*\{?(\d+)\}?\s*\(([^()]*)\)", r" (\1(\3))**\2 ", s)  # \cos^3(x)
    s = re.sub(r"\\" + fn + r"\b", r" \1", s)
    s = s.replace("\\", " ").replace("{", "(").replace("}", ")").replace("[", "(").replace("]", ")")
    s = s.replace("^", "**")
    s = re.sub(r"\binfinity\b|\binf\b", "oo", s, flags=re.I)
    s = re.sub(r"(?<=\d),(?=\d{3}(\D|$))", "", s)  # 1,000 -> 1000 (only inside a single value)
    return s.strip()


_SYMPY = None


def _sympy():
    global _SYMPY
    if _SYMPY is None:
        import sympy
        from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                                implicit_multiplication_application, convert_xor)
        tr = standard_transformations + (implicit_multiplication_application, convert_xor)
        local = {n: sympy.Symbol(n) for n in ["N", "O", "S", "Q", "C", "I", "i", "E", "beta", "gamma", "lambda", "x", "y", "z", "t"]}
        local["i"] = sympy.Symbol("i"); local["I"] = sympy.Symbol("i")   # imaginary unit: compare symbolically
        local["e"] = sympy.E; local["pi"] = sympy.pi; local["oo"] = sympy.oo
        local["ln"] = sympy.log
        _SYMPY = (sympy, parse_expr, tr, local)
    return _SYMPY


def _parse(s):
    sympy, parse_expr, tr, local = _sympy()
    s = _latex_to_expr(s)
    if not s or len(s) > 400:
        return None
    try:
        with time_limit(2):
            e = parse_expr(s, transformations=tr, local_dict=dict(local), evaluate=True)
        return e if isinstance(e, sympy.Basic) else None
    except BaseException:
        return None


def _num_close(a, b):
    return abs(a - b) <= max(REL_TOL * abs(b), ABS_TOL)


def _expr_equal(p, g):
    sympy, *_ = _sympy()
    try:
        with time_limit(3):
            if p == g:
                return True
            free = sorted(p.free_symbols | g.free_symbols, key=str)
            if not free:
                pv, gv = complex(sympy.N(p)), complex(sympy.N(g))
                if any(map(math.isnan, [pv.real, gv.real])):
                    return False
                return _num_close(pv.real, gv.real) and _num_close(pv.imag, gv.imag)
            # symbolic: random numeric spot checks, then simplify as tie-breaker
            rng = random.Random(0)
            ok = 0
            for _ in range(4):
                sub = {s: rng.uniform(0.5, 2.5) for s in free}
                pv, gv = complex(sympy.N(p.subs(sub))), complex(sympy.N(g.subs(sub)))
                if not (_num_close(pv.real, gv.real) and _num_close(pv.imag, gv.imag)):
                    return False
                ok += 1
            return ok == 4
    except BaseException:
        try:
            with time_limit(3):
                return sympy.simplify(p - g) == 0
        except BaseException:
            return False


def _is_letters(s):
    return re.fullmatch(r"[A-P]+", s) is not None


def _clean_letters(s):
    t = re.sub(r"\\text\{|\}|[\s\(\)\.\,\*\$]", "", str(s)).upper()
    return t if _is_letters(t) else None


def _is_word(s):
    return re.fullmatch(r"[A-Za-z][A-Za-z \-']*", s.strip()) is not None and len(s.strip()) > 1


def _bracketed(s):
    s = s.strip()
    return len(s) >= 2 and s[0] in "([" and s[-1] in ")]"


def answer_equal(pred, gold):
    """One blank: predicted string vs gold string."""
    if pred is None:
        return False
    p, g = _strip_latex(pred), _strip_latex(gold)
    inf = lambda t: re.sub(r"(?i)\binfinity\b|\binf\b", "oo", t)
    p, g = inf(p), inf(g)
    if p == g or p.replace(" ", "").lower() == g.replace(" ", "").lower():
        return True
    if g in ("T", "F") and p.lower() in ("true", "false"):   # True/False questions
        return p[0].upper() == g
    gl = g.strip() if re.fullmatch(r"[A-P]{1,4}", g.strip()) else None
    if gl is not None:                                    # letter answer(s), uppercase golds only
        pl = _clean_letters(p)
        return pl is not None and sorted(set(pl)) == sorted(set(gl))
    if _is_word(g) and not re.search(r"\d", g) and len(g) > 2 and not re.fullmatch(r"[a-z]", g):
        if _is_word(p):
            return p.strip().lower() == g.strip().lower()
    # equations: compare right-hand sides
    if "=" in g and not any(op in g for op in ["<=", ">="]):
        g = g.split("=")[-1]
        if "=" in p:
            p = p.split("=")[-1]
    elif "=" in p and not any(op in p for op in ["<=", ">="]):
        p = p.split("=")[-1]
    # tuples / intervals: bracket types must match, then element-wise
    if _bracketed(g) and "," in g:
        if not _bracketed(p) or p.strip()[0] != g.strip()[0] or p.strip()[-1] != g.strip()[-1]:
            return False
        gp, pp = split_top_level(g.strip()[1:-1]), split_top_level(p.strip()[1:-1])
        return len(gp) == len(pp) and all(answer_equal(a, b) for a, b in zip(pp, gp))
    pe, ge = _parse(p), _parse(g)
    if pe is None or ge is None:
        return False
    return _expr_equal(pe, ge)


def extract_mcq_letter(text):
    b = last_boxed(text)
    if b is not None:
        m = re.search(r"\b([A-P])\b", _strip_latex(b).upper()) or re.fullmatch(r"\(?([A-P])\)?\.?", _strip_latex(b).upper())
        if m:
            return m.group(1)
    tail = final_section(text) or ""
    m = re.findall(r"answer\s*(?:is|:)\s*\(?\**\s*([A-P])\b", tail, flags=re.I)
    return m[-1].upper() if m else None


def pred_parts(text, n_gold):
    """Split the final boxed answer into one string per blank."""
    b = last_boxed(text)
    if b is None:
        return None
    if n_gold == 1:
        return [b]
    parts = split_top_level(b)
    # the OTHER prompt tells the model to write letters together (\boxed{BC}) -> one letter per blank
    if len(parts) == 1 and _clean_letters(parts[0]) and len(_clean_letters(parts[0])) == n_gold:
        parts = list(_clean_letters(parts[0]))
    return parts


def grade(text, item):
    """Returns (correct: bool, frac: float in [0,1]). Mirrors 'last \\boxed{}' extraction."""
    golds = gold_list(item)
    if item.get("options"):
        ok = extract_mcq_letter(text) == golds[0].strip().upper()
        return ok, float(ok)
    parts = pred_parts(text, len(golds))
    if parts is None or len(parts) != len(golds):
        return False, 0.0
    hits = sum(answer_equal(p, g) for p, g in zip(parts, golds))
    return hits == len(golds), hits / len(golds)


# ----------------------------------------------------------------------------
# RL reward (uses the same grader; strict about finishing and formatting)
# ----------------------------------------------------------------------------
R_CORRECT, R_PARTIAL_MAX, R_WRONG, R_NO_ANSWER = 1.0, 0.5, 0.0, -0.5


def reward_for(text, item, truncated):
    final = final_section(text)
    if truncated or final is None or last_boxed(final) is None:
        return R_NO_ANSWER
    ok, frac = grade(final, item)
    if ok:
        return R_CORRECT
    if len(gold_list(item)) > 1 and frac > 0:
        return R_PARTIAL_MAX * frac
    return R_WRONG


def summarize(rows, label=""):
    by = {}
    for r in rows:
        by.setdefault(r["qtype"], []).append(r["correct"])
    allc = [r["correct"] for r in rows]
    out = {"label": label, "n": len(rows), "overall": 100 * sum(allc) / max(1, len(allc))}
    for t in ["mcq", "free_single", "free_multi"]:
        v = by.get(t, [])
        out[t] = 100 * sum(v) / len(v) if v else None
    out["truncated_pct"] = 100 * sum(r.get("finish_reason") == "length" for r in rows) / max(1, len(rows))
    return out


# ----------------------------------------------------------------------------
# Comparing runs honestly: paired differences on the same questions
# ----------------------------------------------------------------------------
def per_question(results_path, items, which):
    """{id: correct} for one results file, re-graded with the current grader, restricted to one split."""
    split, by = make_split(items), {it["id"]: it for it in items}
    out = {}
    for r in load_jsonl(results_path):
        it = by.get(r["id"])
        if it is not None and split[it["id"]] == which:
            out[r["id"]] = bool(grade(r["response"], it)[0])
    return out


def paired_diff(a, b, iters=4000, seed=0):
    """Accuracy of b minus a (points) on questions both answered, with a 95% bootstrap CI over questions.
    With 125-250 questions, single-sample scores move several points from sampling luck alone; this says
    whether a difference is bigger than that."""
    ids = sorted(set(a) & set(b), key=str)
    n = len(ids)
    if n == 0:
        return None
    d = [float(b[i]) - float(a[i]) for i in ids]   # per-question accuracy (0/1, or a mean over samples)
    rng = random.Random(seed)
    boots = sorted(100.0 * sum(d[rng.randrange(n)] for _ in range(n)) / n for _ in range(iters))
    return {"n": n, "diff": 100.0 * sum(d) / n, "lo": boots[int(0.025 * iters)], "hi": boots[int(0.975 * iters) - 1],
            "fixed": sum(x > 0 for x in d), "broke": sum(x < 0 for x in d)}


# ----------------------------------------------------------------------------
# Compute-unit guard for long steps (survives disconnects: time used is stored next to the outputs)
# ----------------------------------------------------------------------------
class TimeBudget:
    def __init__(self, path, max_hours):
        self.path, self.max_s = path, (max_hours or 0) * 3600
        self.used0 = 0.0
        if os.path.exists(path):
            try:
                self.used0 = float(json.load(open(path))["seconds"])
            except Exception:
                pass
        import time
        self._time, self.t0 = time.time, time.time()

    def used(self):
        return self.used0 + self._time() - self.t0

    def save(self):
        tmp = self.path + ".tmp"
        with open(tmp, "w") as f:
            json.dump({"seconds": self.used()}, f)
        os.replace(tmp, self.path)

    def would_exceed(self, next_s):
        """True if doing one more unit of work (estimated next_s seconds) would pass the limit."""
        return self.max_s > 0 and self.used() + next_s > self.max_s

In [ ]:
%%writefile /content/qwen_ft/mathrl/run_eval.py
"""Evaluate a model (optionally + LoRA adapter) on a split with the frozen protocol.

python -m mathrl.run_eval --split test --out RESULTS/pe_test.jsonl
python -m mathrl.run_eval --split test --lora ADAPTER_DIR --out RESULTS/sft_test.jsonl
Resumable: rerunning skips questions already in --out.
"""
import argparse, json, os, time
from mathrl.common import (BASE_MODEL, EVAL_MAX_TOKENS, EVAL_SAMPLING, LORA_RANK, load_jsonl, make_split,
                           prompt_text, qtype, gold_list, grade, done_keys, append_jsonl, summarize)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--split", default="test", choices=["test", "dev", "train", "all"])
    ap.add_argument("--base", default=BASE_MODEL, help="HF id or local merged-model dir")
    ap.add_argument("--lora", default=None)
    ap.add_argument("--out", required=True)
    ap.add_argument("--label", default="")
    ap.add_argument("--max_tokens", type=int, default=EVAL_MAX_TOKENS)
    ap.add_argument("--chunk", type=int, default=128, help="questions per saved chunk")
    ap.add_argument("--gpu_util", type=float, default=0.90)
    ap.add_argument("--seed", type=int, default=0, help="sampling seed; extra test samples use 1, 2, 3 ...")
    ap.add_argument("--prompts", choices=["pe", "starter"], default="pe", help="system prompts: PE (default) or the original baseline's")
    a = ap.parse_args()

    items = load_jsonl(a.data)
    split = make_split(items)
    sel = [it for it in items if a.split == "all" or split[it["id"]] == a.split]
    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    done = done_keys(a.out)
    todo = [it for it in sel if it["id"] not in done]
    print(f"[eval] split={a.split} n={len(sel)} done={len(done & {i['id'] for i in sel})} todo={len(todo)} lora={a.lora}")

    if todo:
        from transformers import AutoTokenizer
        from vllm import LLM, SamplingParams
        tok = AutoTokenizer.from_pretrained(BASE_MODEL)          # one tokenizer/template for every model
        prompts = {it["id"]: prompt_text(tok, it, a.prompts) for it in todo}
        max_prompt = max(len(tok(p, add_special_tokens=False)["input_ids"]) for p in prompts.values())
        max_len = max_prompt + a.max_tokens + 64
        print(f"[eval] longest prompt {max_prompt} tok -> max_model_len {max_len}")
        kw = dict(model=a.base, tokenizer=BASE_MODEL, max_model_len=max_len, gpu_memory_utilization=a.gpu_util,
                  seed=a.seed, enable_prefix_caching=True)
        lora_req = None
        if a.lora:
            from vllm.lora.request import LoRARequest
            kw.update(enable_lora=True, max_lora_rank=max(LORA_RANK, 64), max_loras=1)
            lora_req = LoRARequest("adapter", 1, os.path.abspath(a.lora))
        llm = LLM(**kw)
        sp = SamplingParams(n=1, max_tokens=a.max_tokens, **EVAL_SAMPLING)
        t0 = time.time()
        for s in range(0, len(todo), a.chunk):
            chunk = todo[s:s + a.chunk]
            outs = llm.generate([prompts[it["id"]] for it in chunk], sp, lora_request=lora_req)
            rows = []
            for it, o in zip(chunk, outs):
                c = o.outputs[0]
                ok, frac = grade(c.text, it)
                rows.append({"id": it["id"], "split": split[it["id"]], "qtype": qtype(it), "gold": gold_list(it),
                             "response": c.text, "n_tokens": len(c.token_ids), "finish_reason": c.finish_reason,
                             "correct": ok, "frac": frac})
            append_jsonl(a.out, rows)
            print(f"[eval] saved {s + len(chunk)}/{len(todo)}  ({(time.time() - t0) / 60:.1f} min)", flush=True)

    # Always re-grade from the file so every model is scored by the same, current grader.
    by_id = {it["id"]: it for it in sel}
    rows = []
    for r in load_jsonl(a.out):
        if r["id"] in by_id:
            r["correct"], r["frac"] = grade(r["response"], by_id[r["id"]])
            rows.append(r)
    rows = list({r["id"]: r for r in rows}.values())
    s = summarize(rows, a.label or os.path.basename(a.out))
    with open(a.out.replace(".jsonl", "_summary.json"), "w") as f:
        json.dump(s, f, indent=1)
    print("[eval] RESULT", json.dumps(s))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/qwen_ft/mathrl/gen_samples.py
"""Rejection sampling: the base model answers each training problem K times.

One pass gives us both
  * SFT data  - the model's own correct, finished solutions (exact token ids kept)
  * RL data   - per-problem pass rates, used to keep problems that are neither always nor never solved
Problems = our TRAIN split (never dev/test) + optional MMLU-Pro 10-option MCQs (same format as our MCQs).
Resumable: rerun skips problems already in --out.
--max_tokens should equal the SFT length cap: a solution longer than that can't be used for SFT,
and GRPO rolls out with an even smaller budget, so generating past it only burns compute.
"""
import argparse, json, os, random, time
from mathrl.common import (TimeBudget, BASE_MODEL, EVAL_MAX_TOKENS, EVAL_SAMPLING, load_jsonl, make_split, prompt_text,
                           qtype, gold_list, grade, final_section, last_boxed, done_keys, append_jsonl, NearDupIndex)

MMLU_CATS = {"math", "physics", "engineering", "business"}


def mmlu_items(n, public_items, seed=0):
    if n <= 0:
        return []
    from datasets import load_dataset
    ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test")
    dup = NearDupIndex(public_items)
    pool, n_dup = [], 0
    for r in ds:
        if str(r["category"]).lower() not in MMLU_CATS:
            continue
        opts = [o for o in r["options"] if str(o).strip() and str(o).strip() != "N/A"]
        if len(opts) != len(r["options"]):
            continue  # keep only clean 10-option items
        if dup.is_dup(r["question"], r["options"]):
            n_dup += 1
            continue  # near-copy of one of our questions: never train on it
        pool.append({"id": f"mmlupro_{r['question_id']}", "question": r["question"], "options": list(r["options"]),
                     "answer": r["answer"], "src": "mmlupro"})
    print(f"[gen] MMLU-Pro: {len(pool)} usable, {n_dup} skipped as near-copies of our questions")
    random.Random(seed).shuffle(pool)
    return pool[:n]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True, help="samples.jsonl (one row per problem)")
    ap.add_argument("--items_out", required=True, help="problems.jsonl (all problems incl. external)")
    ap.add_argument("--k", type=int, default=4)
    ap.add_argument("--mmlu_n", type=int, default=800)
    ap.add_argument("--max_tokens", type=int, default=EVAL_MAX_TOKENS)
    ap.add_argument("--chunk", type=int, default=64)
    ap.add_argument("--gpu_util", type=float, default=0.90)
    ap.add_argument("--max_hours", type=float, default=0, help="stop cleanly before this much GPU time (total over resumes); 0 = no limit")
    a = ap.parse_args()

    public = load_jsonl(a.data)
    split = make_split(public)
    saved = load_jsonl(a.items_out) if os.path.exists(a.items_out) else []
    if not saved or sum(p["src"] == "mmlupro" for p in saved) < a.mmlu_n:
        # MMLU-Pro selection is a deterministic shuffle, so a larger MMLU_N extends the saved list
        train = [dict(it, src="train") for it in public if split[it["id"]] == "train"]
        saved = train + mmlu_items(a.mmlu_n, public)
        with open(a.items_out, "w") as f:
            for p in saved:
                f.write(json.dumps(p) + "\n")
    # a smaller MMLU_N on a resumed run just stops early (the problems file keeps every item already sampled)
    mm = [p for p in saved if p["src"] == "mmlupro"][:max(a.mmlu_n, 0)]
    problems = [p for p in saved if p["src"] != "mmlupro"] + mm
    assert not any(split.get(p["id"]) in ("dev", "test") for p in problems), "dev/test leaked into training problems"
    done = done_keys(a.out)
    todo = [p for p in problems if p["id"] not in done]
    print(f"[gen] problems={len(problems)} (train={sum(p['src']=='train' for p in problems)}, "
          f"mmlupro={sum(p['src']=='mmlupro' for p in problems)}) done={len(done)} todo={len(todo)} k={a.k}")
    if not todo:
        return
    if a.max_hours and TimeBudget(a.out + ".time.json", a.max_hours).would_exceed(0):
        print(f"[gen] time limit of {a.max_hours} h already used up; keeping the {len(done)} sampled problems. "
              "Raise the limit to sample more.")
        return

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    eos_ids = {tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")}
    prompts = {p["id"]: prompt_text(tok, p) for p in todo}
    max_prompt = max(len(tok(t, add_special_tokens=False)["input_ids"]) for t in prompts.values())
    llm = LLM(model=BASE_MODEL, max_model_len=max_prompt + a.max_tokens + 64, gpu_memory_utilization=a.gpu_util,
              seed=0, enable_prefix_caching=True)
    sp = SamplingParams(n=a.k, max_tokens=a.max_tokens, **EVAL_SAMPLING)
    t0 = time.time()
    budget = TimeBudget(a.out + ".time.json", a.max_hours)
    last_chunk_s = 0.0
    for s in range(0, len(todo), a.chunk):
        if s and budget.would_exceed(last_chunk_s):
            print(f"[gen] STOPPED at the time limit ({a.max_hours} h of GPU time used). {len(done) + s} problems have samples; "
                  f"{len(todo) - s} were skipped (train questions go first, so these are extra MMLU-Pro ones). "
                  "Continue with the next step, or raise the limit and re-run to sample more.", flush=True)
            break
        tc = time.time()
        chunk = todo[s:s + a.chunk]
        outs = llm.generate([prompts[p["id"]] for p in chunk], sp)
        rows = []
        for p, o in zip(chunk, outs):
            samples = []
            for c in o.outputs:
                ok, frac = grade(c.text, p)
                fin = final_section(c.text)
                # "clean" = finished thinking, single final answer after </think>, and that answer is correct
                clean = (c.finish_reason == "stop" and fin is not None and last_boxed(fin) is not None
                         and grade(fin, p)[0])
                ids = [t for t in c.token_ids]
                while ids and ids[-1] in eos_ids:
                    ids.pop()
                samples.append({"correct": ok, "frac": frac, "clean": clean, "n_tokens": len(c.token_ids),
                                "finish_reason": c.finish_reason,
                                "completion_ids": ids if clean else None,
                                "text": c.text if clean else c.text[-1500:]})
            rows.append({"id": p["id"], "src": p["src"], "qtype": qtype(p), "k": a.k,
                         "n_correct": sum(x["correct"] for x in samples), "samples": samples})
        append_jsonl(a.out, rows)
        last_chunk_s = time.time() - tc
        budget.save()
        el = (time.time() - t0) / 60
        print(f"[gen] {s + len(chunk)}/{len(todo)} problems  {el:.1f} min  "
              f"(~{el / (s + len(chunk)) * (len(todo) - s - len(chunk)):.0f} min left)", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/qwen_ft/mathrl/build_sft.py
"""Build the SFT dataset (token ids + loss mask), all in the exact inference format.

Sources
  1. In-domain self-distillation: the model's own *clean* correct solutions to our TRAIN problems
     (and MMLU-Pro MCQs), shortest first -> teaches correct multi-blank formatting + finishing on time.
  2. Teacher distillation: DeepSeek-R1 step-by-step solutions from open-r1/OpenR1-Math-220k, kept only if
     verified correct, complete, our grader agrees with the final \\boxed{}, and short enough.
Each example = chat-template prompt (same system prompts as eval) + completion; loss only on the completion.
"""
import argparse, json, random, statistics
from mathrl.common import (BASE_MODEL, load_jsonl, prompt_text, grade, final_section, last_boxed, NearDupIndex)

R1_TYPES = {"Algebra", "Calculus", "Inequalities", "Number Theory", "Combinatorics", "Other"}


def openr1_examples(n, tok, max_completion, public_items, seed=0, scan_limit=60000):
    if n <= 0:
        return []
    from datasets import load_dataset
    ds = load_dataset("open-r1/OpenR1-Math-220k", "default", split="train", streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=5000)
    out, scanned, seen_qt, seen_pt, n_dup = [], 0, {}, {}, 0
    dup = NearDupIndex(public_items)
    for r in ds:
        scanned += 1
        seen_qt[r.get("question_type")] = seen_qt.get(r.get("question_type"), 0) + 1
        seen_pt[r.get("problem_type")] = seen_pt.get(r.get("problem_type"), 0) + 1
        if scanned > scan_limit or len(out) >= n:
            break
        if r.get("question_type") != "math-word-problem" or r.get("problem_type") not in R1_TYPES:
            continue
        ans = str(r.get("answer") or "").strip()
        if not ans or len(ans) > 60:
            continue
        if dup.is_dup(r["problem"]):
            n_dup += 1
            continue
        gens = r.get("generations") or []
        ver = r.get("correctness_math_verify") or []
        comp = r.get("is_reasoning_complete") or [True] * len(gens)
        item = {"id": f"openr1_{r.get('uuid', scanned)}", "question": r["problem"], "answer": [ans]}
        best = None
        for g, ok, done in zip(gens, ver, comp):
            if not (ok and done) or not g.startswith("<think>") or "</think>" not in g:
                continue
            completion = g[len("<think>"):]
            completion = completion[1:] if completion.startswith("\n") else completion   # prompt already ends '<think>\n'
            fin = final_section(completion)
            if fin is None or last_boxed(fin) is None or not grade(fin, item)[0]:
                continue  # our grader must agree, so SFT never teaches a format we score as wrong
            if best is None or len(completion) < len(best):
                best = completion
        if best is None:
            continue
        ids = tok(best, add_special_tokens=False)["input_ids"]
        if len(ids) > max_completion:
            continue
        out.append({"source": "openr1", "id": item["id"], "prompt": prompt_text(tok, item), "completion_ids": ids})
    print(f"[sft] OpenR1: kept {len(out)} after scanning {scanned} rows ({n_dup} skipped as near-copies of our questions)")
    if len(out) < 0.25 * n:
        print(f"[sft] WARNING: few OpenR1 examples kept. question_type seen: {seen_qt}  problem_type seen: {seen_pt}")
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--samples", required=True)
    ap.add_argument("--problems", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--per_problem", type=int, default=2, help="max solutions kept per in-domain problem")
    ap.add_argument("--indomain_repeat", type=int, default=2, help="epochs over in-domain data")
    ap.add_argument("--openr1_n", type=int, default=1500)
    ap.add_argument("--max_completion", type=int, default=10240)
    a = ap.parse_args()

    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    problems = {p["id"]: p for p in load_jsonl(a.problems)}
    public_items = load_jsonl(a.data)

    indomain = []
    for row in load_jsonl(a.samples):
        p = problems[row["id"]]
        clean = sorted([s for s in row["samples"] if s["clean"] and s["completion_ids"]
                        and len(s["completion_ids"]) <= a.max_completion], key=lambda s: s["n_tokens"])
        seen, kept = set(), 0
        for s in clean:
            key = hash(tuple(s["completion_ids"]))  # exact duplicates only
            if key in seen:
                continue
            seen.add(key)
            indomain.append({"source": p["src"], "id": p["id"], "prompt": prompt_text(tok, p),
                             "completion_ids": s["completion_ids"]})
            kept += 1
            if kept >= a.per_problem:
                break
    n_in = len(indomain)
    ext = openr1_examples(a.openr1_n, tok, a.max_completion, public_items)
    data = indomain * a.indomain_repeat + ext
    random.Random(0).shuffle(data)
    with open(a.out, "w") as f:
        for d in data:
            f.write(json.dumps(d) + "\n")
    lens = [len(d["completion_ids"]) for d in data] or [0]
    stats = {"indomain_examples": n_in, "indomain_problems": len({d["id"] for d in indomain}),
             "openr1_examples": len(ext), "total_rows": len(data),
             "completion_tokens_mean": int(statistics.mean(lens)), "completion_tokens_max": max(lens),
             "total_train_tokens": sum(lens)}
    with open(a.out.replace(".jsonl", "_stats.json"), "w") as f:
        json.dump(stats, f, indent=1)
    print("[sft] DATASET", json.dumps(stats))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/qwen_ft/mathrl/train_sft.py
"""LoRA SFT on Qwen3-4B-Thinking-2507 (the same model as the baseline), loss on completion tokens only.

Checkpoints go to --ckpt_dir (put it on Drive); rerunning resumes from the latest checkpoint.
"""
import argparse, glob, json, os, random


def make_rows(tok, data, max_seq):
    """prompt ids (masked) + completion ids + <|im_end|> (trained). Exactly what the model sees/produces."""
    eos = tok.convert_tokens_to_ids("<|im_end|>")
    rows, dropped = [], 0
    for d in data:
        p_ids = tok(d["prompt"], add_special_tokens=False)["input_ids"]
        c_ids = list(d["completion_ids"]) + [eos]
        if len(p_ids) + len(c_ids) > max_seq:
            dropped += 1
            continue
        rows.append({"input_ids": p_ids + c_ids, "labels": [-100] * len(p_ids) + c_ids})
    return rows, dropped


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sft_data", required=True)
    ap.add_argument("--ckpt_dir", required=True)
    ap.add_argument("--adapter_out", required=True)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--grad_accum", type=int, default=16)
    ap.add_argument("--max_seq", type=int, default=16384)
    ap.add_argument("--save_steps", type=int, default=25)
    a = ap.parse_args()

    from unsloth import FastLanguageModel          # must be imported before transformers
    import torch
    from transformers import Trainer, TrainingArguments
    from mathrl.common import BASE_MODEL, LORA_RANK, load_jsonl

    model, tok = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=a.max_seq, load_in_4bit=False, load_in_8bit=False,
        dtype=torch.bfloat16, fast_inference=False)
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=3407)

    eos = tok.convert_tokens_to_ids("<|im_end|>")
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.convert_tokens_to_ids("<|endoftext|>")
    rows, dropped = make_rows(tok, load_jsonl(a.sft_data), a.max_seq)
    assert rows, "empty SFT dataset"
    n_lab = sum(sum(1 for x in r["labels"] if x != -100) for r in rows)
    print(f"[sft] {len(rows)} examples ({dropped} dropped as too long), {n_lab/1e6:.2f}M trained tokens")
    r0 = rows[0]
    n_prompt = r0["labels"].index(next(x for x in r0["labels"] if x != -100))
    print("[sft] prompt/completion boundary ->", repr(tok.decode(r0["input_ids"][max(0, n_prompt - 12):n_prompt + 12])))

    class DS(torch.utils.data.Dataset):
        def __len__(self): return len(rows)
        def __getitem__(self, i): return rows[i]

    def collate(batch):
        L = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), L), pad, dtype=torch.long)
        lab = torch.full((len(batch), L), -100, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for i, b in enumerate(batch):
            n = len(b["input_ids"])
            ids[i, :n] = torch.tensor(b["input_ids"]); lab[i, :n] = torch.tensor(b["labels"]); att[i, :n] = 1
        return {"input_ids": ids, "labels": lab, "attention_mask": att}

    # Memory check: one forward+backward on the longest example before training, so an out-of-memory
    # error shows up in the first minute instead of hours into the run. Gradients are discarded.
    longest = max(rows, key=lambda r: len(r["input_ids"]))
    batch = {k: v.to(model.device) for k, v in collate([longest]).items()}
    model.train()
    model(**batch).loss.backward()
    model.zero_grad(set_to_none=True)
    del batch
    torch.cuda.empty_cache()
    print(f"[sft] memory check passed on the longest example ({len(longest['input_ids'])} tokens), "
          f"peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB", flush=True)

    args = TrainingArguments(
        output_dir=a.ckpt_dir, per_device_train_batch_size=1, gradient_accumulation_steps=a.grad_accum,
        num_train_epochs=1, learning_rate=a.lr, lr_scheduler_type="cosine", warmup_ratio=0.05,
        weight_decay=0.0, max_grad_norm=1.0, optim="adamw_8bit", bf16=True, logging_steps=5,
        save_strategy="steps", save_steps=a.save_steps, save_total_limit=2, report_to="none", seed=3407,
        dataloader_num_workers=0, remove_unused_columns=False, gradient_checkpointing=False)
    trainer = Trainer(model=model, args=args, train_dataset=DS(), data_collator=collate)
    ckpts = sorted(glob.glob(os.path.join(a.ckpt_dir, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[1]))
    resume = ckpts[-1] if ckpts else None
    print(f"[sft] resume_from={resume}")
    trainer.train(resume_from_checkpoint=resume)
    model.save_pretrained(a.adapter_out)
    tok.save_pretrained(a.adapter_out)
    with open(os.path.join(a.adapter_out, "train_log.json"), "w") as f:
        json.dump(trainer.state.log_history, f)
    print(f"[sft] DONE adapter saved to {a.adapter_out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/qwen_ft/mathrl/merge_lora.py
"""Merge the SFT LoRA into the base weights so GRPO starts from the SFT model (plain transformers + peft)."""
import argparse, os


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--adapter", required=True)
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    if os.path.exists(os.path.join(a.out, "config.json")):
        print(f"[merge] already merged at {a.out}"); return
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    from mathrl.common import BASE_MODEL
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="cuda")
    model = PeftModel.from_pretrained(base, a.adapter).merge_and_unload()
    model.save_pretrained(a.out, safe_serialization=True)
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(a.out)
    print(f"[merge] DONE -> {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/qwen_ft/mathrl/build_grpo.py
"""Pick RL prompts where the reward will actually vary (GRPO learns nothing from all-right/all-wrong groups).

  learnable : solved by 1..K-1 of the K base-model samples
  too long  : solved every time, but answers ran long -> RL can teach finishing inside the budget
  shuffled  : learnable MCQs again with options re-ordered (new correct letter) -> no letter memorization
"""
import argparse, json, random
from mathrl.common import BASE_MODEL, LETTERS, load_jsonl, prompt_text, qtype


def shuffle_mcq(p, rng):
    perm = list(range(len(p["options"])))
    while perm == sorted(perm):
        rng.shuffle(perm)
    gold_idx = LETTERS.index(str(p["answer"]).strip().upper())
    return dict(p, id=f"{p['id']}_shuf", options=[p["options"][i] for i in perm], answer=LETTERS[perm.index(gold_idx)])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--samples", required=True)
    ap.add_argument("--problems", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--max_prompt", type=int, default=2048)
    ap.add_argument("--max_completion", type=int, default=8192)
    ap.add_argument("--long_n", type=int, default=80)
    ap.add_argument("--require_fit", action="store_true",
                    help="keep a learnable problem only if one of its correct samples fits in --max_completion "
                         "(use with train_grpo --cutoff ignore, where longer rollouts teach nothing)")
    ap.add_argument("--mmlu_share", type=float, default=0.35, help="max share of RL prompts from MMLU-Pro (keeps RL focused on our own questions)")
    a = ap.parse_args()
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    problems = {p["id"]: p for p in load_jsonl(a.problems)}
    rng = random.Random(0)
    learn, long_, n_nofit = [], [], 0
    for r in load_jsonl(a.samples):
        p, k, c = problems[r["id"]], r["k"], r["n_correct"]
        if 0 < c < k:
            if a.require_fit and not any(s["correct"] and s["n_tokens"] < a.max_completion for s in r["samples"]):
                n_nofit += 1
                continue
            learn.append(p)
        elif c == k and min(s["n_tokens"] for s in r["samples"]) > 0.75 * a.max_completion:
            long_.append(p)
    rng.shuffle(long_)
    longest = max((s["n_tokens"] for r in load_jsonl(a.samples) for s in r["samples"]), default=0)
    if longest < 0.9 * a.max_completion:
        print(f"[grpo-data] NOTE: samples were generated with at most ~{longest} tokens, below the RL budget of {a.max_completion}. "
              "Problems that need more than that to solve are missing from the RL set; for 16k RL set SAMPLE_MAX_TOKENS = 16384 "
              "in notebook 2 before sampling.")
    chosen = learn + long_[:a.long_n] + [shuffle_mcq(p, rng) for p in learn if p.get("options")]
    own = [p for p in chosen if p.get("src") != "mmlupro"]
    ext = [p for p in chosen if p.get("src") == "mmlupro"]
    rng.shuffle(ext)
    cap = int(len(own) * a.mmlu_share / (1 - a.mmlu_share))
    n_ext_dropped = max(0, len(ext) - cap)
    chosen = own + ext[:cap]
    rows, skipped = [], 0
    for p in chosen:
        text = prompt_text(tok, p)
        if len(tok(text, add_special_tokens=False)["input_ids"]) > a.max_prompt:
            skipped += 1
            continue
        item = {k: p[k] for k in ("id", "question", "answer") if k in p}
        if p.get("options"):
            item["options"] = p["options"]
        rows.append({"prompt": text, "item": json.dumps(item), "qtype": qtype(p)})
    rng.shuffle(rows)
    with open(a.out, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"[grpo-data] learnable={len(learn)} (dropped {n_nofit} whose correct answers were all longer than {a.max_completion} tokens) long={min(len(long_), a.long_n)} "
          f"shuffled_mcq={sum(1 for p in learn if p.get('options'))} -> {len(rows)} prompts "
          f"({skipped} skipped: prompt > {a.max_prompt} tokens; {n_ext_dropped} MMLU-Pro prompts dropped by the {a.mmlu_share:.0%} cap)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/qwen_ft/mathrl/train_grpo.py
"""GRPO on top of the SFT model. Reward = the same grader used for evaluation.

Design (see notebook for reasoning):
  * reward: +1 correct, partial credit on multi-blank, 0 wrong, -0.5 finished without a final \\boxed{} answer
  * loss_type=dr_grpo (no length bias), beta=0 (no KL / no reference model), clip-higher eps_high=0.28
  * rollouts cut off at the RL budget (8k) are, by default, left out of the loss (DAPO "overlong filtering"):
    eval allows 16k, so RL must not teach the model to cut short reasoning that would have finished by 16k.
    --cutoff penalize instead gives them -0.5 (pushes for shorter answers).
  * prompts pre-filtered so each group has a real mix of right and wrong answers
Checkpoints every --save_steps to --ckpt_dir (Drive); rerun resumes.
"""
import argparse, glob, json, os, shutil

os.environ.setdefault("UNSLOTH_VLLM_STANDBY", "1")


def make_reward_fn(max_completion, stats, cutoff="ignore"):
    from mathrl.common import reward_for, R_WRONG

    def correctness_reward(prompts, completions, completion_ids=None, item=None, **kw):
        out = []
        for i, c in enumerate(completions):
            text = c if isinstance(c, str) else c[-1]["content"]
            truncated = completion_ids is not None and len(completion_ids[i]) >= max_completion
            if truncated and cutoff == "ignore":
                r = R_WRONG          # neutral for the group baseline; trl drops it from the loss (mask_truncated_completions)
            else:
                r = reward_for(text, json.loads(item[i]), truncated)
            out.append(r)
            stats["n"] += 1; stats["correct"] += r == 1.0; stats["no_answer"] += r < 0; stats["cut_off"] += truncated
        return out
    return correctness_reward


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True, help="merged SFT model dir (or base model id to skip SFT)")
    ap.add_argument("--prompts", required=True)
    ap.add_argument("--ckpt_dir", required=True)
    ap.add_argument("--adapter_out", required=True)
    ap.add_argument("--steps", type=int, default=120)
    ap.add_argument("--lr", type=float, default=5e-6)
    ap.add_argument("--num_generations", type=int, default=8)
    ap.add_argument("--prompts_per_step", type=int, default=2)
    ap.add_argument("--max_prompt", type=int, default=2048)
    ap.add_argument("--max_completion", type=int, default=8192)
    ap.add_argument("--temperature", type=float, default=0.8)
    ap.add_argument("--save_steps", type=int, default=10)
    ap.add_argument("--keep_steps", default="20,40", help="comma list of steps whose adapter is kept for dev eval")
    ap.add_argument("--gpu_util", type=float, default=0.85)
    ap.add_argument("--cutoff", choices=["ignore", "penalize"], default="ignore",
                    help="rollouts that hit --max_completion: leave out of the loss (default) or reward -0.5")
    ap.add_argument("--max_hours", type=float, default=0, help="stop cleanly before this much training time (total over resumes); 0 = no limit")
    a = ap.parse_args()

    from unsloth import FastLanguageModel           # before trl/transformers
    import torch
    from datasets import Dataset
    from transformers import TrainerCallback
    from trl import GRPOConfig, GRPOTrainer
    import time
    from mathrl.common import LORA_RANK, load_jsonl, TimeBudget

    model, tok = FastLanguageModel.from_pretrained(
        model_name=a.base, max_seq_length=a.max_prompt + a.max_completion, load_in_4bit=False,
        load_in_8bit=False, dtype=torch.bfloat16, fast_inference=True, max_lora_rank=LORA_RANK,
        gpu_memory_utilization=a.gpu_util)
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=3407)

    rows = load_jsonl(a.prompts)
    need = a.steps * a.prompts_per_step
    print(f"[grpo] {len(rows)} prompts available, {need} needed for {a.steps} steps"
          + ("" if len(rows) >= need else " -> some prompts will repeat (more than 1 epoch)"))
    ds = Dataset.from_list(rows)

    stats = {"n": 0, "correct": 0, "no_answer": 0, "cut_off": 0}
    correctness_reward = make_reward_fn(a.max_completion, stats, a.cutoff)

    keep = {int(s) for s in a.keep_steps.split(",") if s.strip()}
    os.makedirs(a.ckpt_dir, exist_ok=True)
    budget = TimeBudget(os.path.join(a.ckpt_dir, "time_used.json"), a.max_hours)
    clock = {"last": time.time(), "times": []}
    hist, zstd = [], []     # per-step (acc, no_answer, cut_off) and trl's share of groups with no reward spread

    def health(step, per_step):
        """Plain-language check every 10 steps; only warns, never stops the run."""
        h = hist[-10:]
        acc, na, co = (sum(x[i] for x in h) / len(h) for i in range(3))
        zs = sum(zstd[-10:]) / len(zstd[-10:]) if zstd else None
        msg = f"[grpo] HEALTH @ step {step}: last {len(h)} steps acc={acc:.2f} no_answer={na:.2f} cut_off={co:.2f}"
        if zs is not None:
            msg += f" groups_without_signal={zs:.2f}"
        print(msg, flush=True)
        if len(hist) >= 20:
            first = sum(x[0] for x in hist[:10]) / 10
            print(f"[grpo]   accuracy: first 10 steps {first:.2f} -> last 10 steps {acc:.2f} (about +/-0.04 of noise)", flush=True)
        warn = []
        if zs is not None and zs > 0.6:
            warn.append("most groups have no reward spread, so little is being learned (prompts too easy or too hard)")
        if co > 0.35:
            warn.append("over a third of answers are cut off at the token limit")
        if len(hist) >= 20 and acc < sum(x[0] for x in hist[:10]) / 10 - 0.12:
            warn.append("training accuracy fell clearly; consider stopping and keeping the SFT model")
        if per_step / 60 > 14:
            warn.append(f"{per_step / 60:.0f} min/step means fewer than {int(a.max_hours * 60 / (per_step / 60)) if a.max_hours else 0} steps fit in the limit")
        for w in warn:
            print("[grpo]   WARNING: " + w, flush=True)

    class Keep(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kw):
            if logs and "frac_reward_zero_std" in logs:
                zstd.append(float(logs["frac_reward_zero_std"]))

        def on_train_begin(self, args, state, control, **kw):
            clock["last"] = time.time()   # step timing excludes model loading / vLLM warm-up

        def on_step_end(self, args, state, control, **kw):
            if state.global_step in keep:
                d = os.path.join(os.path.dirname(a.adapter_out.rstrip("/")), f"grpo_adapter_step{state.global_step}")
                model.save_pretrained(d); tok.save_pretrained(d)
                print(f"[grpo] kept adapter for dev eval: {d}", flush=True)
            if stats["n"]:
                print(f"[grpo] step {state.global_step}: acc={stats['correct']/stats['n']:.2f} "
                      f"no_answer={stats['no_answer']/stats['n']:.2f} cut_off={stats['cut_off']/stats['n']:.2f} "
                      f"over {stats['n']} samples", flush=True)
                hist.append((stats["correct"] / stats["n"], stats["no_answer"] / stats["n"], stats["cut_off"] / stats["n"]))
                stats.update(n=0, correct=0, no_answer=0, cut_off=0)
            now = time.time()
            clock["times"].append(now - clock["last"]); clock["last"] = now
            budget.save()
            recent = clock["times"][-5:]
            per_step = sum(recent) / len(recent)
            left = state.max_steps - state.global_step
            if state.global_step % 5 == 0 or len(clock["times"]) == 3:
                print(f"[grpo] {per_step / 60:.1f} min/step -> ~{left * per_step / 3600:.1f} h for the remaining {left} steps "
                      f"({budget.used() / 3600:.1f} h used" + (f" of {a.max_hours} h limit)" if a.max_hours else ")"), flush=True)
            if hist and state.global_step % 10 == 0:
                health(state.global_step, per_step)
            if left > 0 and budget.would_exceed(per_step):
                print(f"[grpo] STOPPING at step {state.global_step}: the next step would pass the {a.max_hours} h limit. "
                      "The adapter is saved as the final one.", flush=True)
                control.should_training_stop = True
            return control

    cfg = GRPOConfig(
        output_dir=a.ckpt_dir, max_steps=a.steps, learning_rate=a.lr, lr_scheduler_type="constant_with_warmup",
        # constant LR (as in DAPO / Dr. GRPO): stopping early at the unit limit leaves no half-finished schedule
        warmup_steps=5, weight_decay=0.0, max_grad_norm=1.0, optim="adamw_torch", adam_beta2=0.99,
        bf16=True, logging_steps=1, save_strategy="steps", save_steps=a.save_steps, save_total_limit=2,
        report_to="none", seed=3407,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=a.num_generations * a.prompts_per_step,
        num_generations=a.num_generations,
        max_prompt_length=a.max_prompt, max_completion_length=a.max_completion,
        temperature=a.temperature, top_p=0.95, top_k=None, min_p=0.0,
        loss_type="dr_grpo", beta=0.0, epsilon=0.2, epsilon_high=0.28,
        scale_rewards=False, mask_truncated_completions=(a.cutoff == "ignore"),
    )
    trainer = GRPOTrainer(model=model, processing_class=tok, reward_funcs=[correctness_reward],
                          args=cfg, train_dataset=ds, callbacks=[Keep()])
    ckpts = sorted(glob.glob(os.path.join(a.ckpt_dir, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[1]))
    resume = ckpts[-1] if ckpts else None
    print(f"[grpo] resume_from={resume}")
    trainer.train(resume_from_checkpoint=resume)
    model.save_pretrained(a.adapter_out)
    tok.save_pretrained(a.adapter_out)
    with open(os.path.join(a.adapter_out, "train_log.json"), "w") as f:
        json.dump(trainer.state.log_history, f)
    print(f"[grpo] DONE adapter saved to {a.adapter_out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/qwen_ft/mathrl/recheck_grpo.py
"""Re-measure the RL prompts with the SFT model (the model RL actually starts from).

build_grpo picks prompts from the *base* model's pass rates. SFT moves those: some become always-right
(a group of 8 identical rewards teaches nothing), some stay always-wrong. Here the merged SFT model answers
each candidate K times at the RL temperature and budget, and we keep the prompts it gets right sometimes but
not always (plus, when cut-offs are penalized, always-right ones whose answers run long).
Resumable; stops cleanly at --max_hours (unchecked prompts are then kept as they were).
"""
import argparse, json, os, time
from mathrl.common import (BASE_MODEL, TimeBudget, load_jsonl, append_jsonl, done_keys, reward_for)


def select(rows, checks, k, keep_long, max_tokens, min_keep):
    """rows: prompt rows from build_grpo; checks: {id: {"n_correct", "min_correct_tokens"}}."""
    keep, stats = [], {"unchecked": 0, "mixed": 0, "long": 0, "all_right": 0, "all_wrong": 0}
    for r in rows:
        c = checks.get(json.loads(r["item"])["id"])
        if c is None:
            keep.append(r); stats["unchecked"] += 1
        elif 0 < c["n_correct"] < k:
            keep.append(r); stats["mixed"] += 1
        elif c["n_correct"] == k and keep_long and (c["min_correct_tokens"] or 0) > 0.75 * max_tokens:
            keep.append(r); stats["long"] += 1
        elif c["n_correct"] == k:
            stats["all_right"] += 1
        else:
            stats["all_wrong"] += 1
    if len(keep) < min_keep:
        print(f"[recheck] only {len(keep)} prompts left (< {min_keep}); keeping the original set instead")
        return rows, stats
    return keep, stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="merged SFT model dir")
    ap.add_argument("--prompts_in", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--k", type=int, default=4)
    ap.add_argument("--max_tokens", type=int, default=16384)
    ap.add_argument("--temperature", type=float, default=0.8)
    ap.add_argument("--keep_long", action="store_true")
    ap.add_argument("--min_keep", type=int, default=60)
    ap.add_argument("--chunk", type=int, default=64)
    ap.add_argument("--gpu_util", type=float, default=0.90)
    ap.add_argument("--max_hours", type=float, default=0)
    a = ap.parse_args()

    rows = load_jsonl(a.prompts_in)
    checks_path = a.out.replace(".jsonl", "_checks.jsonl")
    done = done_keys(checks_path)
    todo = [r for r in rows if json.loads(r["item"])["id"] not in done]
    budget = TimeBudget(checks_path + ".time.json", a.max_hours)
    print(f"[recheck] {len(rows)} prompts, {len(done)} already checked, {len(todo)} to go")

    if todo and not budget.would_exceed(0):
        from vllm import LLM, SamplingParams
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(BASE_MODEL)
        max_prompt = max(len(tok(r["prompt"], add_special_tokens=False)["input_ids"]) for r in todo)
        llm = LLM(model=a.model, tokenizer=BASE_MODEL, max_model_len=max_prompt + a.max_tokens + 64,
                  gpu_memory_utilization=a.gpu_util, seed=0, enable_prefix_caching=True)
        sp = SamplingParams(n=a.k, temperature=a.temperature, top_p=0.95, min_p=0.0, max_tokens=a.max_tokens)
        t0, last = time.time(), 0.0
        for s in range(0, len(todo), a.chunk):
            if s and budget.would_exceed(last):
                print(f"[recheck] STOPPED at the time limit; {len(todo) - s} prompts stay unchecked (kept as is)")
                break
            tc = time.time()
            chunk = todo[s:s + a.chunk]
            outs = llm.generate([r["prompt"] for r in chunk], sp)
            res = []
            for r, o in zip(chunk, outs):
                item = json.loads(r["item"])
                ok = [(reward_for(c.text, item, c.finish_reason == "length") == 1.0, len(c.token_ids)) for c in o.outputs]
                res.append({"id": item["id"], "n_correct": sum(x for x, _ in ok),
                            "min_correct_tokens": min((n for x, n in ok if x), default=None)})
            append_jsonl(checks_path, res)
            last = time.time() - tc
            budget.save()
            el = (time.time() - t0) / 60
            print(f"[recheck] {s + len(chunk)}/{len(todo)}  {el:.1f} min (~{el / (s + len(chunk)) * (len(todo) - s - len(chunk)):.0f} min left)", flush=True)

    checks = {c["id"]: c for c in load_jsonl(checks_path)} if os.path.exists(checks_path) else {}
    keep, stats = select(rows, checks, a.k, a.keep_long, a.max_tokens, a.min_keep)
    with open(a.out, "w") as f:
        for r in keep:
            f.write(json.dumps(r) + "\n")
    print(f"[recheck] kept {len(keep)}/{len(rows)} prompts  {json.dumps(stats)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/qwen_ft/mathrl/report.py
"""One results table: every run re-graded with the same grader on the same questions."""
import argparse, glob, json, os
from mathrl.common import load_jsonl, make_split, grade, qtype, summarize, paired_diff


def sample_files(path):
    """The main results file plus any extra-sample files written next to it (pe_test_s1.jsonl, ...)."""
    return [path] + sorted(glob.glob(path.replace(".jsonl", "_s*.jsonl")))


def score_file(path, items_by_id, split, which, label):
    n_expected = sum(1 for v in split.values() if v == which)
    files = sample_files(path)
    all_rows, per_q, complete = [], {}, True
    for fp in files:
        rows = {}
        for r in load_jsonl(fp):
            it = items_by_id.get(r["id"])
            if it is None or split[it["id"]] != which:
                continue
            ok, frac = grade(r["response"], it)
            rows[r["id"]] = {"id": r["id"], "qtype": qtype(it), "correct": ok, "frac": frac,
                             "finish_reason": r.get("finish_reason", "length" if "</think>" not in r["response"] else "stop")}
        complete &= len(rows) == n_expected
        all_rows += rows.values()
        for i, r in rows.items():
            per_q.setdefault(i, []).append(float(r["correct"]))
    s = summarize(all_rows, label)            # with k samples: average accuracy over all k x n answers
    s["n"] = len(per_q)
    s["k"] = len(files)
    s["complete"] = complete and len(per_q) == n_expected
    s["per_q"] = {i: sum(v) / len(v) for i, v in per_q.items()}
    return s


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--runs", nargs="+", required=True, help="label=path.jsonl ...")
    ap.add_argument("--split", default="test")
    a = ap.parse_args()
    items = load_jsonl(a.data)
    by_id = {it["id"]: it for it in items}
    split = make_split(items)
    print(f"\n{'run':<34}{'n':>5}{'MCQ':>8}{'single':>8}{'multi':>8}{'OVERALL':>9}{'trunc%':>8}")
    print("-" * 80)
    scored = []
    for spec in a.runs:
        label, path = spec.split("=", 1)
        if not os.path.exists(path):
            print(f"{label:<34}  (not run yet)"); continue
        s = score_file(path, by_id, split, a.split, label)
        f = lambda v: f"{v:7.1f}%" if v is not None else "    n/a"
        flag = ("" if s["complete"] else "  <- incomplete") + (f"  (avg of {s['k']} samples)" if s["k"] > 1 else "")
        print(f"{label:<34}{s['n']:>5}{f(s['mcq'])}{f(s['free_single'])}{f(s['free_multi'])}{s['overall']:8.1f}%{s['truncated_pct']:7.1f}%{flag}")
        scored.append((label, s))
    # Paired comparisons on the same questions: is a change bigger than sampling luck?
    pairs = [(scored[i - 1], scored[i]) for i in range(1, len(scored))]
    labels = [l for l, _ in scored]
    if "PE" in labels:                                  # also PE -> each later stage (e.g. PE -> PE+SFT+RL)
        i = labels.index("PE")
        pairs += [(scored[i], x) for x in scored[i + 2:]]
    if pairs:
        print(f"\npaired change on the same questions (95% CI; 'fixed'/'broke' = questions that flipped):")
        for (la, sa), (lb, sb) in pairs:
            c = paired_diff(sa["per_q"], sb["per_q"])
            if c:
                verdict = "clear gain" if c["lo"] > 0 else ("clear loss" if c["hi"] < 0 else "within noise")
                print(f"  {la} -> {lb}: {c['diff']:+.1f} pts  [{c['lo']:+.1f}, {c['hi']:+.1f}]  "
                      f"fixed {c['fixed']}, broke {c['broke']}  -> {verdict}")
    print(f"\nsplit = {a.split}; grader = mathrl/common.py grade(); baseline used 8,192 max tokens, all others 16,384.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/qwen_ft/tests/test_grader.py
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from mathrl.common import answer_equal, grade, reward_for, split_top_level, last_boxed, extract_mcq_letter, R_CORRECT, R_NO_ANSWER, R_WRONG

SHOULD_MATCH = [
    ("\\dfrac{x\\sqrt{36 - x^2}}{x^2 - 18}", "2*x*sqrt(6^2-x^2)/(2*x^2-6^2)"),
    ("0.928", "2.947*1.26/[sqrt(16)]"),
    ("8\\cos^3(x) + 6\\cos^2(x) - 6", "8*[cos(x)]^3+6*[cos(x)]^2-6"),
    ("True", "T"), ("false", "F"),
    ("\\frac{\\sqrt{3}}{3}", "1/sqrt(3)"),
    ("12.08", "12.0814"),                 # 0.01% off -> within 0.1%
    ("(-1, -3)", "(-1,-3)"),
    ("negative", "NEGATIVE"),
    ("\\text{neither}", "neither"),
    ("0.076x", "7.6/100*x"),
    ("5\\left(\\frac{7}{4}\\right)^x", "5*1.75**x"),
    ("y = 2x+1", "2*x+1"),
    ("2x+1", "y=2*x+1"),
    ("1,000", "1000"),
    ("(-8, \\infty)", "(-8,infinity)"),
    ("\\frac{\\pi}{2}", "1.5707963"),
    ("e^{2}", "7.389056"),
    ("BC", "BC"), ("C,F", "CF"),
]
SHOULD_NOT_MATCH = [
    ("2.33", "2.32624773420025"),         # 0.16% off -> outside tolerance
    ("(1, 3)", "[1,3]"),                   # interval bracket types differ
    ("-0.770", "ln(0.462)"),
    ("x+1", "x-1"),
    ("B", "C"),
    ("increasing", "decreasing"),
    ("\\frac{7}{4}, -\\frac{7}{4}", "(1.75, -1.75)"),   # point needs parentheses
    ("", "5"),
    ("x", "A"),                            # lowercase variable is not an option letter
]

def check():
    bad = []
    for p, g in SHOULD_MATCH:
        if not answer_equal(p, g): bad.append(("expected MATCH", p, g))
    for p, g in SHOULD_NOT_MATCH:
        if answer_equal(p, g): bad.append(("expected NO match", p, g))
    # splitting
    assert split_top_level("(1, 2), 3, [4,5]") == ["(1, 2)", "3", "[4,5]"], split_top_level("(1, 2), 3, [4,5]")
    assert split_top_level("a; b, c") == ["a", "b, c"]
    assert last_boxed("x \\boxed{\\frac{1}{2}} y \\boxed{3, \\{4\\}}") == "3, \\{4\\}"
    # full responses
    mcq = {"question": "q", "options": ["1"] * 10, "answer": "C"}
    assert grade("...</think>\n\nSo \\boxed{C}", mcq) == (True, 1.0)
    assert grade("...</think>\n\n\\boxed{\\text{(C)}}", mcq)[0]
    assert grade("I think the answer is D maybe", mcq) == (False, 0.0)      # no box, no 'answer is' in final part
    multi = {"question": "a [ANS] b [ANS]", "answer": ["3", "(1,2)"]}
    assert grade("</think> \\boxed{3, (1, 2)}", multi) == (True, 1.0)
    assert grade("</think> \\boxed{3, (1, 5)}", multi) == (False, 0.5)
    assert grade("</think> \\boxed{3}", multi) == (False, 0.0)               # wrong count -> no credit
    letters = {"question": "A. x B. y [ANS] [ANS]", "answer": ["G", "B"]}
    assert grade("</think> \\boxed{GB}", letters)[0] and grade("</think> \\boxed{G, B}", letters)[0]
    # reward
    assert reward_for("reasoning</think>\n\n\\boxed{3, (1,2)}", multi, truncated=False) == R_CORRECT
    assert reward_for("reasoning \\boxed{3, (1,2)}", multi, truncated=False) == R_NO_ANSWER   # never closed </think>
    assert reward_for("reasoning</think>\\boxed{3, (1,2)}", multi, truncated=True) == R_NO_ANSWER
    assert reward_for("reasoning</think>\\boxed{4, (9,9)}", multi, truncated=False) == R_WRONG
    assert 0 < reward_for("r</think>\\boxed{3, (9,9)}", multi, truncated=False) < R_CORRECT
    return bad

if __name__ == "__main__":
    bad = check()
    for b in bad: print("FAIL", b)
    print("grader tests:", "ALL PASS" if not bad else f"{len(bad)} FAIL")
    sys.exit(1 if bad else 0)

In [ ]:
run("python tests/test_grader.py")
run(f"python -c \"from mathrl.common import *; items=load_jsonl('{DATA}'); s=make_split(items); import collections; print(collections.Counter(s.values()))\"")

In [ ]:
assert os.path.exists(f"{WORK}/sft/adapter/adapter_model.safetensors") and os.path.exists(f"{WORK}/sft/samples.jsonl"), "Run notebook 2 first"
assert RATE == 7.52, f"This notebook needs an A100 80GB (this runtime: {GPU_NAME}). Disconnect and reconnect an A100 runtime until nvidia-smi shows 80GB."
print(f"RL: rollouts up to {MAX_COMPLETION} tokens; re-check limit {RECHECK_UNITS} units, GRPO limit {GRPO_UNITS} units = {hours_for(GRPO_UNITS)} h on {GPU_NAME}")

## Step 1 · Merge the SFT adapter into the weights (RL starts from the SFT model)

In [ ]:
run(f"python -m mathrl.merge_lora --adapter {WORK}/sft/adapter --out {MERGED}", done_if=f"{MERGED}/config.json")

## Step 2 · Pick RL prompts with real reward variance

In [ ]:
run(f"python -m mathrl.build_grpo --samples {WORK}/sft/samples.jsonl --problems {WORK}/sft/problems.jsonl --out {WORK}/grpo/prompts.jsonl --max_completion {MAX_COMPLETION}", done_if=f"{WORK}/grpo/prompts.jsonl")

## Step 2b · Re-check those prompts with the SFT model
The prompts above were chosen from the *base* model's answers; SFT changes which ones it gets right. Keep only prompts the SFT model gets right sometimes but not always, so every RL group carries signal.

In [ ]:
run(f"python -m mathrl.recheck_grpo --model {MERGED} --prompts_in {WORK}/grpo/prompts.jsonl --out {WORK}/grpo/prompts_checked.jsonl --max_tokens {MAX_COMPLETION} --keep_long --max_hours {hours_for(RECHECK_UNITS)}", done_if=f"{WORK}/grpo/prompts_checked.jsonl")

## Step 3 · GRPO (checkpoints every 10 steps; re-run resumes)
Watch the `[grpo] step N: acc=… no_answer=… cut_off=…` lines: accuracy should drift up and `cut_off` (answers that hit 16k) down. In trl's logs, `frac_reward_zero_std` near 1.0 means most groups had no signal.

In [ ]:
grpo = lambda util: run(f"python -m mathrl.train_grpo --base {MERGED} --prompts {WORK}/grpo/prompts_checked.jsonl --ckpt_dir {WORK}/grpo/ckpt --adapter_out {WORK}/grpo/adapter --steps {GRPO_STEPS} --lr {GRPO_LR} --prompts_per_step {PROMPTS_PER_STEP} --max_completion {MAX_COMPLETION} --keep_steps 20,40 --gpu_util {util} --max_hours {hours_for(GRPO_UNITS)} --cutoff {CUT_OFF_ROLLOUTS}", done_if=f"{WORK}/grpo/adapter/adapter_model.safetensors")
try:
    grpo(GRPO_GPU_UTIL)
except StepFailed as e:
    if not e.oom: raise
    print("\n>>> Out of GPU memory: retrying automatically with less memory for generation (resumes from the last checkpoint).\n")
    grpo(0.70)

## Step 4 · Choose the RL checkpoint on dev, then score test

In [ ]:
cands = {"step20": f"{WORK}/grpo/grpo_adapter_step20", "step40": f"{WORK}/grpo/grpo_adapter_step40", "final": f"{WORK}/grpo/adapter"}
dev = {}
for name, path in cands.items():
    if os.path.exists(f"{path}/adapter_model.safetensors"):
        out = f"{WORK}/results/rl_{name}_dev.jsonl"
        run(f"python -m mathrl.run_eval --data {DATA} --split dev --base {MERGED} --lora {path} --label 'RL {name}' --out {out}", done_if=out.replace(".jsonl", "_summary.json"))
        dev[name] = summary(out)["overall"]
best = max(dev, key=dev.get)
print("dev:", {k: f"{v:.1f}%" for k, v in dev.items()}, "| chosen:", best)
sys.path.insert(0, CODE)
from mathrl.common import load_jsonl, per_question, paired_diff
sft_dev = f"{WORK}/results/sft_dev.jsonl"
if os.path.exists(sft_dev):
    items = load_jsonl(DATA)
    c = paired_diff(per_question(sft_dev, items, "dev"), per_question(f"{WORK}/results/rl_{best}_dev.jsonl", items, "dev"))
    print(f"DEV  PE+SFT -> PE+SFT+RL: change {c['diff']:+.1f} pts (95% CI {c['lo']:+.1f} to {c['hi']:+.1f}; fixed {c['fixed']}, broke {c['broke']})")
    if c["hi"] < 0:
        print("RL is clearly worse than SFT on dev. Test is still scored so you have the number, but report PE+SFT as the main result; "
              "a retry would use GRPO_LR = 3e-6 (delete grpo/ckpt + grpo/adapter and re-run step 3).")
    elif c["diff"] < 0:
        print("Slightly below SFT, but within the noise of 125 dev questions. Go ahead to test.")
BEST_RL = cands[best]

In [ ]:
run(f"python -m mathrl.run_eval --data {DATA} --split test --base {MERGED} --lora {BEST_RL} --label 'PE + SFT + RL' --out {WORK}/results/rl_test.jsonl", done_if=f"{WORK}/results/rl_test_summary.json")

In [ ]:
#@title Results table (same grader, same questions for every row)
import shutil
for S in ["dev", "test"]:
    rl = f"{WORK}/results/rl_{S}.jsonl"
    if S == "dev" and not os.path.exists(rl) and "best" in globals():
        shutil.copy(f"{WORK}/results/rl_{best}_dev.jsonl", rl)
    run(f'python -m mathrl.report --data {DATA} --split {S} --runs "Baseline (starter prompts, 8k)={BASELINE}" "Baseline (starter prompts, 16k)={WORK}/results/base16k_{S}.jsonl" "PE={WORK}/results/pe_{S}.jsonl" "PE + SFT={WORK}/results/sft_{S}.jsonl" "PE + SFT + RL={WORK}/results/rl_{S}.jsonl"')

## Step 5 · More precise test scores
With one answer per question, a 250-question score moves ±6 points from sampling luck alone, so a real +3 from SFT could look like noise.
This re-answers the 250 test questions 3 more times with **all three** models (PE, PE+SFT, PE+SFT+RL), each with a new seed, so every row of the final table is an average of 4 answers per question.
Same prompts, settings and grader as before; only the random seed differs. Dev and training are untouched.

In [ ]:
#@title Extra test samples + final table
for i in range(1, EXTRA_TEST_SAMPLES + 1):
    for tag, model in [("pe", ""), ("sft", f"--lora {WORK}/sft/adapter"), ("rl", f"--base {MERGED} --lora {BEST_RL}")]:
        out = f"{WORK}/results/{tag}_test_s{i}.jsonl"
        run(f"python -m mathrl.run_eval --data {DATA} --split test --seed {i} {model} --label '{tag} sample {i}' --out {out}",
            done_if=out.replace(".jsonl", "_summary.json"))
if EXTRA_TEST_SAMPLES:
    S = "test"
    run(f'python -m mathrl.report --data {DATA} --split test --runs "Baseline (starter prompts, 8k)={BASELINE}" "Baseline (starter prompts, 16k)={WORK}/results/base16k_{S}.jsonl" "PE={WORK}/results/pe_{S}.jsonl" "PE + SFT={WORK}/results/sft_{S}.jsonl" "PE + SFT + RL={WORK}/results/rl_{S}.jsonl"')

### If something goes wrong
* **Disconnected / runtime reset:** run all cells again; the merge is redone locally (~5 min) and GRPO resumes from its last Drive checkpoint.
* **Out of GPU memory in GRPO:** handled automatically (one retry with less generation memory, resuming from the checkpoint).
* **`[grpo] STOPPING ... limit`:** the adapter at that step is used as the final one; the dev comparison of the saved checkpoints still runs.
* **Not an A100 80GB:** Runtime → Disconnect and delete runtime, then reconnect with A100 until the first cell shows 80GB.